In [ ]:
import pandas as pd
import os

In [ ]:
os.getcwd()

In [ ]:
# read the analyzed metadata excel
metadata = pd.read_excel('BERTopic_topics_metadata_checked.xlsx', index_col=0)
print(metadata.shape)
metadata.head()


In [ ]:
# # get Topic and Topic_new
# metadata_topics = metadata[['Topic', 'Name', 'Topic_new', '2nd level topics', '3rd level topics', 'Representation', 'Representative_Docs']].drop_duplicates()
# print(metadata_topics.shape)

In [ ]:
# read articles
articles = pd.read_csv('NOS_covid_BERTtopics_parlevel.csv', sep=';', header=0, index_col=0)

In [ ]:
print(articles.shape)
articles.head()

In [ ]:
articles.columns
# drop subtopic_ columns
cols = [c for c in articles.columns if 'subtopic_' not in c]
articles = articles[cols]
articles.head()

In [ ]:
# see the dates min and max
print(articles['Date'].min())
print(articles['Date'].max())

In [ ]:
# get the number of unique article_id per date
articles['Date'] = pd.to_datetime(articles['Date'])

articles['Date'].value_counts().sort_index()

In [ ]:
# drop articles after 31-05-2022
articles = articles[articles['Date'] <= '2022-05-31']
print(articles.shape)

In [ ]:
articles.head()

In [ ]:
# merge articles with topics
articles = articles.merge(metadata, how='left', left_on='new_topics', right_on='Topic')
print(articles.shape)

In [ ]:
# see nr of unique articlee_id
articles['article_id'].nunique()

# Plot the nr of articles per week

In [ ]:
# get the weeknumber
articles['week'] = articles['Date'].dt.isocalendar().week
articles['year'] = articles['Date'].dt.isocalendar().year

# Create a proper datetime for sorting
# First day of the week for each year-week combination
# Use pandas to_datetime with format %Y-%U-%w (Year-Week-Weekday)
articles['week_start'] = pd.to_datetime(articles['year'].astype(str) + '-' + 
                                        articles['week'].astype(str) + '-1', 
                                        format='%Y-%W-%w')

# Create the year_week as before for display purposes
articles['year_week'] = articles['year'].astype(str) + '-' + articles['week'].astype(str)

print(articles['year_week'].nunique())
print(articles['year_week'].value_counts(dropna=False).sort_index())

In [ ]:
# Calculate number of unique articles per week, but sort by week_start
weekly_nr_articles = (articles.sort_values(by='week_start')
                      .groupby(['week_start', 'year_week'])['article_id']
                      .nunique()
                      .reset_index())

weekly_nr_articles.columns = ['week_start', 'year_week', 'nr_articles']

In [ ]:
# plot the number of articles per week
import matplotlib.pyplot as plt
import seaborn as sns
# Plot using the chronologically sorted data
plt.figure(figsize=(12, 6))
sns.set_style("whitegrid")
sns.set_palette("rocket")

# Sort by week_start but use year_week for display
weekly_nr_articles = weekly_nr_articles.sort_values('week_start')

sns.lineplot(data=weekly_nr_articles, x='year_week', y='nr_articles', color='red', marker='o')

# Improve x-axis labels by showing fewer ticks
# Only show every nth tick to avoid overcrowding
n = max(1, len(weekly_nr_articles) // 30)  # Show ~15 ticks
plt.xticks(range(0, len(weekly_nr_articles), n), 
           weekly_nr_articles['year_week'].iloc[::n], 
           rotation=45)

plt.title('Number of Articles Covering the Pandemic per Week on NOS.nl', fontsize=16)
plt.xlabel('Year-weeknr', fontsize=12)
plt.ylabel('Number of articles', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
articles = articles.drop(columns=['new_topics'])

In [ ]:
articles.head()
# make a new column called representative text, which is the first item in the list of representative docs
articles['Representative_Text'] = articles['Representative_Docs'].apply(lambda x: x.split('||')[0])
articles.head()

In [ ]:
articles.columns

In [ ]:
print(articles['2nd Level Topic Name'].nunique())
print(articles['2nd Level Topic Name'].value_counts(dropna=False))

In [ ]:
print(articles['3rd Level Topic Name'].nunique())
print(articles['3rd Level Topic Name'].value_counts(dropna=False))

In [ ]:
# get the count of articles per topic by counting unique article_id per topic
articles_per_topic = articles.groupby('3rd Level Topic Name')['article_id'].nunique().reset_index()
articles_per_topic.columns = ['3rdleveltopics', 'n_articles']
print(articles_per_topic.shape)
articles_per_topic=articles_per_topic.sort_values('n_articles', ascending=False)

In [ ]:
articles_per_topic

In [ ]:
# make a barplot of the number of articles per topic
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(12, 6))
sns.barplot(data=articles_per_topic, x='n_articles', y='3rdleveltopics', palette='rocket')
plt.xlabel('Number of articles')
plt.ylabel('Sub-topics')
plt.title('Number of articles per sub-topic')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
unique_nr_articles = articles['article_id'].nunique()
print(f"Number of unique articles: {unique_nr_articles}")

In [ ]:
# # make percentages - BERTOPIC
# articles_per_topic['percentage'] = (articles_per_topic['n_articles'] / unique_nr_articles) * 100

# plt.figure(figsize=(10, 12))
# sns.barplot(data=articles_per_topic, x='percentage', y='3rdleveltopics', palette='rocket')
# plt.xlim(0, 50)
# # show percentage values
# for i, v in enumerate(articles_per_topic['percentage']):
#     plt.text(v + 0.5, i + .1, str(round(v, 1)) + '%', color='black', fontsize=18)

# # wrap the y-axis labels
# plt.yticks(fontsize=20, rotation=0, ha='right')

# plt.xlabel('Percentage of articles covering the sub-topic', fontsize=20)
# # plt.ylabel('Sub-topics', fontsize=20)
# # hide the y-label
# plt.ylabel('')
# plt.show()
# import matplotlib as mpl
# mpl.rcParams['figure.dpi'] = 300

In [ ]:
articles_per_topic['percentage'] = (articles_per_topic['n_articles'] / unique_nr_articles) * 100
import matplotlib as mpl
# Set global parameters for APA 7 compliance
mpl.rcParams['figure.dpi'] = 300
plt.rcParams['font.family'] = 'serif'  # APA recommends serif fonts
plt.rcParams['font.size'] = 12  # Base font size for APA

# Use appropriate figure size for APA (6.5 x 8 inches for Word document)
plt.figure(figsize=(6.5, 8))
sns.barplot(data=articles_per_topic, x='percentage', y='3rdleveltopics', palette='rocket')
plt.xlim(0, 50)

# Add percentage labels
for i, v in enumerate(articles_per_topic['percentage']):
    plt.text(v + 0.5, i, str(round(v, 1)) + '%', color='black', fontsize=12, va='center')

# Format axes
plt.xticks(fontsize=12)
plt.yticks(fontsize=12, rotation=0, ha='right')
plt.xlabel('Percentage of Articles Covering the Sub-topic', fontsize=12, weight='bold')
plt.ylabel('Sub-topics', fontsize=12, weight='bold')

plt.tight_layout()

# Save with journal specifications (300 DPI for color figures)
plt.savefig('analyses/NOS/analyses_for_paper/BERTOPIC_topics.tiff', dpi=300, bbox_inches='tight', format='tiff')
# Alternative formats accepted by journal:
plt.savefig('analyses/NOS/analyses_for_paper/BERTOPIC_topics.jpg', dpi=300, bbox_inches='tight', format='jpeg')
plt.show()

# Topics Coded by LLM's

In [ ]:
# read data
topics_coded_llms = pd.read_csv('NOS_final_topics_v2.csv', sep=';', header=0, index_col=0)
# drop if column has unnamed
topics_coded_llms = topics_coded_llms.loc[:, ~topics_coded_llms.columns.str.contains('^Unnamed')]
# topics_coded_llms.head()

In [ ]:
# see the dates min and max
print(topics_coded_llms['Date'].min())
print(topics_coded_llms['Date'].max())

In [ ]:
# get the number of unique article_id per date
topics_coded_llms['Date'] = pd.to_datetime(topics_coded_llms['Date'])

# topics_coded_llms['Date'].value_counts().sort_index()

In [ ]:
# drop topics_coded_llms after 31-05-2022
topics_coded_llms = topics_coded_llms[topics_coded_llms['Date'] <= '2022-05-31']
print(topics_coded_llms.shape)

In [ ]:
topics_coded_llms.about_covid.value_counts(dropna=False)

In [ ]:
# filter for only articles about covid
topics_coded_llms = topics_coded_llms[topics_coded_llms['about_covid'] == 1]
print(topics_coded_llms.shape)

In [ ]:
# rename subtopics
topics_coded_llms = topics_coded_llms.rename(columns={'subtopic_a_pred': 'Pandemic Statistics and Status Updates',
                                                      'subtopic_b_pred': 'Covid-19 Restrictions and Measures',
                                                      'subtopic_c_pred': 'Covid-19 Tests and Testing Procedures',
                                                      'subtopic_d_pred': 'Covid-19 Vaccines, Vaccination Procedures and Campaigns (incl. 2G & 3G)',
                                                      'subtopic_e_pred': 'Long-Covid and Long-Term Effects of Covid-19 on Health',
                                                      'subtopic_f_pred': 'Healthcare, Medical Response and Challenges',
                                                      'subtopic_g_pred': 'Scientific/Medical Research/Knowledge on Covid-19 and the Coronavirus',
                                                      'subtopic_h_pred': 'Impact of the Pandemic on Economy & Recovery Measures',
                                                      'subtopic_ij_pred': 'Societal Consequences of the Pandemic & Mental Health',
                                                      'subtopic_k_pred': 'Impact of Pandemic on Rights and Liberties',
                                                      'subtopic_l_pred': 'Misinformatie over coronavirus/pandemie & complottheorieën',
                                                      'subtopic_m_pred': 'Impact of Pandemic on Politics and Political Discussions',
                                                      'subtopic_n_pred': 'Global Response & International Collaboration ',
                                                      })


 
topics_coded_llms.columns

In [ ]:
subtopics = ['Pandemic Statistics and Status Updates',
        'Covid-19 Restrictions and Measures',
        'Covid-19 Tests and Testing Procedures',
        'Covid-19 Vaccines, Vaccination Procedures and Campaigns (incl. 2G & 3G)',
        'Long-Covid and Long-Term Effects of Covid-19 on Health',
        'Healthcare, Medical Response and Challenges',
        'Scientific/Medical Research/Knowledge on Covid-19 and the Coronavirus',
        'Impact of the Pandemic on Economy & Recovery Measures',
        'Societal Consequences of the Pandemic & Mental Health',
        'Misinformatie over coronavirus/pandemie & complottheorieën',
        'Impact of Pandemic on Rights and Liberties',
        'Impact of Pandemic on Politics and Political Discussions',
        'Global Response & International Collaboration ']
subtopics_counts = topics_coded_llms[subtopics].sum()
subtopics_counts = subtopics_counts.sort_values(ascending=False)
subtopics_counts

In [ ]:
# make percentages
subtopics_counts = subtopics_counts.reset_index()
subtopics_counts.columns = ['Subtopic', 'n_articles']
subtopics_counts['percentage'] = (subtopics_counts['n_articles'] / 6491) * 100
subtopics_counts

In [ ]:
# drop Misinformatie over coronavirus/pandemie & complottheorieën
subtopics_counts = subtopics_counts[subtopics_counts['Subtopic'] != 'Misinformatie over coronavirus/pandemie & complottheorieën']
subtopics_counts = subtopics_counts.sort_values('percentage', ascending=False)

In [ ]:
plt.figure(figsize=(6.5, 8))
sns.barplot(data=subtopics_counts, x='percentage', y='Subtopic', palette='rocket')
plt.xlim(0, 50)

# Add percentage labels
for i, v in enumerate(subtopics_counts['percentage']):
    plt.text(v + 0.5, i, str(round(v, 1)) + '%', color='black', fontsize=12, va='center')

# Format axes
plt.xticks(fontsize=12)
plt.yticks(fontsize=12, rotation=0, ha='right')
plt.xlabel('Percentage of Articles Covering the Sub-topic', fontsize=12, weight='bold')
plt.ylabel('Sub-topics', fontsize=12, weight='bold')

plt.tight_layout()

# Save with journal specifications (300 DPI for color figures)
plt.savefig('analyses/NOS/analyses_for_paper/LLM_topics.tiff', dpi=300, bbox_inches='tight', format='tiff')
# Alternative formats accepted by journal:
plt.savefig('analyses/NOS/analyses_for_paper/LLM_topics.jpg', dpi=300, bbox_inches='tight', format='jpeg')

plt.show()

In [ ]:
topics_coded_llms

In [ ]:
# count nr topics per article
nr_topics_perarticle_llms = topics_coded_llms.groupby('article_id')[subtopics].sum().reset_index()
nr_topics_perarticle_llms['n_topics'] = nr_topics_perarticle_llms[subtopics].sum(axis=1)
nr_topics_perarticle_llms = nr_topics_perarticle_llms[['article_id', 'n_topics']]
nr_topics_perarticle_llms

In [ ]:
# how many articles have 0 topics
nr_topics_perarticle_llms[nr_topics_perarticle_llms['n_topics'] == 0]

In [ ]:
nr_topics_perarticle_llms['n_topics'].value_counts()

In [ ]:
# merge nr_topics_perarticle_llms with nr_topics_perarticle
nr_topics_perarticle_merged = nr_topics_perarticle.merge(nr_topics_perarticle_llms, how='outer', on='article_id')
# name article_id, topic_bert, topic_llm
nr_topics_perarticle_merged.columns = ['article_id', 'n_topics_bert', 'n_topics_llm']
nr_topics_perarticle_merged


In [ ]:
# get difference between n_topics_bert and n_topics_llm
nr_topics_perarticle_merged['difference'] = nr_topics_perarticle_merged['n_topics_bert'] - nr_topics_perarticle_merged['n_topics_llm']
# make column difference positive, negative, or 0
nr_topics_perarticle_merged['difference'] = nr_topics_perarticle_merged['difference'].apply(lambda x: 'positive' if x > 0 else 'negative' if x < 0 else '0')
nr_topics_perarticle_merged.difference.value_counts()

In [ ]:
5131/(5131+1017+343)

In [ ]:
nr_topics_perarticle_merged[nr_topics_perarticle_merged['n_topics_llm'] == 0].n_topics_bert.value_counts()

In [ ]:
nr_topics_perarticle_merged[nr_topics_perarticle_merged['n_topics_llm'] == 0].n_topics_bert.value_counts()

In [ ]:
nr_topics_perarticle_merged[(nr_topics_perarticle_merged['n_topics_llm'] == 0) & (nr_topics_perarticle_merged['n_topics_bert'] >= 4)]

In [ ]:
# check the topics of articles with 0 topics of llm
article_ids_notopics = nr_topics_perarticle_merged[nr_topics_perarticle_merged['n_topics_llm'] == 0]['article_id']
articles[articles['article_id'].isin(article_ids_notopics)].Topic_new.value_counts()

In [ ]:
topics_coded_llms.columns

In [ ]:
topics_coded_llms.Owner.value_counts()

In [ ]:
topics_coded_llms[topics_coded_llms['article_id'].isin(article_ids_notopics) & (topics_coded_llms['Owner'] == 'NOS Nieuws')].Category.value_counts()

In [ ]:
# see the articles wehre the difference is >- 6
nr_topics_perarticle_merged[nr_topics_perarticle_merged['difference'] >= 5]

# Merge llms with bertopic

In [ ]:
topics_coded_llms.columns
# select article_id and subtopics
topics_coded_llms_selected = topics_coded_llms[['article_id'] + subtopics].reset_index(drop=True)
topics_coded_llms_selected.head()

In [ ]:
# select article_id and topic from articles
articles_selected = articles[['article_id', 'Topic_new']].drop_duplicates().reset_index(drop=True)
articles_selected.head()

In [ ]:
# merge
articles_topics_merged = articles_selected.merge(topics_coded_llms_selected, how='outer', on='article_id')
articles_topics_merged.head()

In [ ]:
articles_topics_merged.shape

In [ ]:
articles_topics_merged.isnull().sum()

In [ ]:
articles_topics_merged['Topic_new'].value_counts()

In [ ]:
# crosstab topic_new with all subtopics
for subtopic in subtopics:
    print(subtopic)
    crosstab = pd.crosstab(articles_topics_merged['Topic_new'], articles_topics_merged[subtopic])
    print(crosstab)